# **Scraping Planos Super Telecom**

_Página do provedor: https://superitelecom.com.br/_

In [1]:
# Bibliotecas
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
from datetime import date
import re

## **1. Extração**

In [2]:
# Obter lista de planos
texto_html = open("../../data/scraped/super.html", encoding="utf-8")
soup = BeautifulSoup(texto_html, "html.parser")
planos = soup.find_all("div", role="article")
print(f"{len(planos)} cartões de planos capturados.")

6 cartões de planos capturados.


In [3]:
# Extrair informações dos planos
def raspar_plano(plano):
    velocidade = plano.find("span", class_="text-6xl").text
    preco = plano.find("span", class_="text-2xl").text
    pre_sva = plano.find_all("span", class_="text-[9px]")
    lista_sva = [i.text for i in pre_sva]
    svas = ", ".join(lista_sva)

    return {"velocidade": velocidade,
            "preco": preco,
            "svas": svas}

dados = []

for i in planos:
    dados_plano = raspar_plano(i)
    dados.append(dados_plano)

df = pd.DataFrame(dados)
df

,velocidade,preco,svas
0,100,"84,90","HBOMax*, Super i Play*, Desconto Energia**"
1,250,"89,90","HBOMax*, Super i Play*, Desconto Energia**"
2,300,"94,90","HBOMax*, Super i Play*, Desconto Energia**"
3,450,"99,90","HBOMax*, Super i Play*, Desconto Energia**"
4,500,"104,90","HBOMax*, Super i Play*, Desconto Energia**"
5,800,"129,90","HBOMax*, Super i Play*, Desconto Energia**"


## **2. Tratamento**

In [4]:
df_tratado = df.copy()

# Data e provedor
df_tratado["DATA"] = date.today().strftime("%Y-%m-%d")
df_tratado["ISP"] = "Super"

# Velocidade
df_tratado["VELOCIDADE"] = df_tratado["velocidade"]

# PRECO
df_tratado["PRECO"] = (
    df_tratado["preco"]
    .astype(str)
    .str.replace(",", ".", regex=False)
    .astype(float)
)

# SVAs
df_tratado["SVAs"] = df_tratado["svas"]

#Seleção de colunas
df_tratado = df_tratado[["DATA", "ISP", "VELOCIDADE", "SVAs", "PRECO"]]

df_tratado

,DATA,ISP,VELOCIDADE,SVAs,PRECO
0,2026-05-12,Super,100,"HBOMax*, Super i Play*, Desconto Energia**",84.9
1,2026-05-12,Super,250,"HBOMax*, Super i Play*, Desconto Energia**",89.9
2,2026-05-12,Super,300,"HBOMax*, Super i Play*, Desconto Energia**",94.9
3,2026-05-12,Super,450,"HBOMax*, Super i Play*, Desconto Energia**",99.9
4,2026-05-12,Super,500,"HBOMax*, Super i Play*, Desconto Energia**",104.9
5,2026-05-12,Super,800,"HBOMax*, Super i Play*, Desconto Energia**",129.9


## **3. Carga**

In [5]:
# Salvando
provedor = "SUPER.csv"
nome_arq = df_tratado["DATA"].max() + "-" + provedor

# Na rede Valenet
pasta = r"\\fileserver\arquivos\Estrategia Comercial\2. POWER BI\ECM - Radar Mercado\ScrapingPlanos"
df_tratado.to_csv(fr"{pasta}\{nome_arq}", index=False)

# Neste Workspace
df_tratado.to_csv(f"../../data/flat/{provedor}", index=False)

print(f"🟢 {nome_arq} salvo!")

🟢 2026-05-12-SUPER.csv salvo!
